# Customer Churn Prediction: Data Cleaning

**Goal:** A telecom company is losing customers. We want to predict which
customers are likely to leave so the company can try to keep them.

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("../data/telco_churn.csv")

In [3]:
df.shape

(7043, 21)

In [4]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

In [6]:
df.describe()

,SeniorCitizen,tenure,MonthlyCharges
count,7043.000000,7043.000000,7043.000000
mean,0.162147,32.371149,64.761692
std,0.368612,24.559481,30.090047
min,0.000000,0.000000,18.250000
25%,0.000000,9.000000,35.500000
50%,0.000000,29.000000,70.350000
75%,0.000000,55.000000,89.850000
max,1.000000,72.000000,118.750000


## First look at the data

1. **Size:** 7,043 customers and 21 columns.
2. **Target:** `Churn` (Yes = the customer left, No = stayed).
3. **Column types:**
   - Numbers: `tenure` (months as a customer), `MonthlyCharges`, `TotalCharges`
   - Categories: gender, Partner, Dependents, services (internet, phone, streaming...), Contract, PaymentMethod, etc.
   - `SeniorCitizen` is stored as 0/1 but is really a Yes/No category.
   - `customerID` is just an ID, not useful for prediction.
4. **Something strange:** `TotalCharges` should be a number, but pandas reads it as text (`str`).
   That's why `describe()` shows only 3 columns instead of 4. Something inside that column is not a number. Investigating next.

In [7]:
total_charges = pd.to_numeric(df["TotalCharges"], errors="coerce")

In [8]:
total_charges.isna().sum()

np.int64(11)

In [9]:
df[total_charges.isna()][["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]]

,customerID,tenure,MonthlyCharges,TotalCharges,Churn
488,4472-LVYGI,0,52.55,,No
753,3115-CZMZD,0,20.25,,No
936,5709-LVOEQ,0,80.85,,No
1082,4367-NUYAO,0,25.75,,No
1340,1371-DWPAZ,0,56.05,,No
3331,7644-OMVMY,0,19.85,,No
3826,3213-VVOLG,0,25.35,,No
4380,2520-SGTTA,0,20.00,,No
5218,2923-ARZLG,0,19.70,,No
6670,4075-WKNIU,0,73.35,,No


In [10]:
df["TotalCharges"] = total_charges.fillna(0)
df["TotalCharges"].dtype

dtype('float64')

In [11]:
df.describe()

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges
count,7043.000000,7043.000000,7043.000000,7043.000000
mean,0.162147,32.371149,64.761692,2279.734304
std,0.368612,24.559481,30.090047,2266.794470
min,0.000000,0.000000,18.250000,0.000000
25%,0.000000,9.000000,35.500000,398.550000
50%,0.000000,29.000000,70.350000,1394.550000
75%,0.000000,55.000000,89.850000,3786.600000
max,1.000000,72.000000,118.750000,8684.800000


### Fixing `TotalCharges`

- `TotalCharges` was stored as text because **11 rows contained a blank space** instead of a number.
- All 11 customers have **tenure = 0**: they had just joined and had not been billed yet.
- So their real total charges are **0**. I filled them with 0 instead of the mean,
  because the mean (~2,280) would wrongly suggest new customers had already paid a lot.
- `TotalCharges` is now a proper number column (`float64`).

In [12]:
df["customerID"].duplicated().sum()

np.int64(0)

In [13]:
df = df.drop(columns=["customerID"])
df.shape

(7043, 20)

In [14]:
df["SeniorCitizen"] = df["SeniorCitizen"].map({0: "No", 1: "Yes"})
df["SeniorCitizen"].value_counts()

SeniorCitizen
No     5901
Yes    1142
Name: count, dtype: int64

In [15]:
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})
df["Churn"].value_counts()

Churn
0    5174
1    1869
Name: count, dtype: int64

In [16]:
df["Churn"].value_counts(normalize=True)

Churn
0    0.73463
1    0.26537
Name: proportion, dtype: float64

In [17]:
df.to_csv("../data/churn_clean.csv", index=False)

## Cleaning summary

- **No duplicate customers** (checked `customerID`).
- **Dropped `customerID`**: an ID can't predict churn and could let the model memorize rows.
- **`SeniorCitizen`** changed from 0/1 to No/Yes, to match the other Yes/No columns.
- **`Churn`** (target) changed to 1 = left, 0 = stayed.
- **Class balance:** 26.5% churned, 73.5% stayed. A model that always predicts
  "no churn" would get 73.5% accuracy while catching zero churners, so
  **accuracy is misleading**. I'll use recall, precision and F1 instead.
- Saved the result as `data/churn_clean.csv` (7,043 rows, 20 columns).